In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Concatenate, Input
from tensorflow.keras.callbacks import EarlyStopping

# --- Set random seeds for reproducibility ---
os.environ['PYTHONHASHSEED'] = '0'
np.random.seed(42)
tf.random.set_seed(42)

# Helper function to downcast numeric data types to save memory
def downcast_dataframe(df):
    """Downcasts numeric data types to save memory."""
    print("Downcasting numeric columns to save memory...")
    for col in df.select_dtypes(include=np.number).columns:
        df[col] = pd.to_numeric(df[col], downcast='float', errors='ignore')
    return df

print("--- Starting Deep Neural Network (DNN) Only Model Training ---")

# --- Load and Merge Data ---
print("Loading and merging data...")
try:
    X_transaction_df = pd.read_csv("../data/processed/train_transaction_sample.csv")
    X_identity_df = pd.read_csv("../data/processed/train_identity_sample.csv")
    y = X_transaction_df['isFraud'].values
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure data files are in the correct directory.")
    exit()

X_df = X_transaction_df.merge(X_identity_df, on="TransactionID", how="left")
print(f"Shape of the merged dataframe: {X_df.shape}")

# --- Prepare Features & Labels ---
# Drop non-tabular data and leaky columns
X_df = X_df.drop(columns=['TransactionID', 'isFraud'], errors="ignore")
leaky_cols = ['DeviceInfo', 'card1', 'id_31', 'id_33', 'Prompt']
X_df = X_df.drop(columns=leaky_cols, errors='ignore')

# Identify and drop high-cardinality categorical columns
HIGH_CARDINALITY_THRESHOLD = 50
categorical_cols_all = X_df.select_dtypes(include=['object', 'category']).columns.tolist()
high_cardinality_cols = [col for col in categorical_cols_all if X_df[col].nunique() > HIGH_CARDINALITY_THRESHOLD]
categorical_cols = [col for col in categorical_cols_all if col not in high_cardinality_cols]
X_df = X_df.drop(columns=high_cardinality_cols, errors='ignore')

# Identify tabular numeric columns
tabular_numeric_cols = X_df.select_dtypes(include=np.number).columns.tolist()

# Ensure all columns are numeric by coercing to float and filling NaNs
for col in tabular_numeric_cols:
    X_df[col] = pd.to_numeric(X_df[col], errors='coerce').fillna(0)

# Preprocessing Pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, tabular_numeric_cols),
        ('cat', categorical_transformer, categorical_cols)],
    remainder='drop',
)

# Split the data and apply preprocessor
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=42
)

# Process the tabular data only
X_train_processed = preprocessor.fit_transform(X_train_df)
X_test_processed = preprocessor.transform(X_test_df)

print("\nFitting and transforming data...")
print(f"Shape of X_train processed tabular data: {X_train_processed.shape}")

# --- DNN Only Model with Keras --- 
num_non_fraud = np.sum(y_train == 0)
num_fraud = np.sum(y_train == 1)
total = num_non_fraud + num_fraud
weight_for_0 = (1 / num_non_fraud) * (total / 2.0)
weight_for_1 = (1 / num_fraud) * (total / 2.0)
class_weight = {0: weight_for_0, 1: weight_for_1}
print(f"Calculated class weights: {class_weight}")

input_tabular = Input(shape=(X_train_processed.shape[1],), name='tabular_input')

# Build the main model
x = Dense(256, activation='relu')(input_tabular)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=input_tabular, outputs=output)

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=[
                  'accuracy',
                  tf.keras.metrics.AUC(name='auc'),
                  tf.keras.metrics.Precision(name='precision'),
                  tf.keras.metrics.Recall(name='recall')
              ])

model.summary()

early_stopping = EarlyStopping(monitor='val_auc', patience=10, mode='max', restore_best_weights=True)

print("\nStarting DNN Only model training...")
history = model.fit(
    X_train_processed,
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test_processed, y_test),
    class_weight=class_weight,
    callbacks=[early_stopping],
    verbose=1
)

print("DNN Only model training complete.")

# --- Evaluation ---
print("\n--- Model Evaluation ---")
probs = model.predict(X_test_processed).flatten()
y_pred = (probs >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
conf_matrix = confusion_matrix(y_test, y_pred)
roc_auc = roc_auc_score(y_test, probs)

print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")
print("Confusion Matrix:")
print(conf_matrix)
print(f"ROC AUC Score: {roc_auc:.3f}")

--- Starting Deep Neural Network (DNN) Only Model Training ---
Loading and merging data...
Shape of the merged dataframe: (50000, 434)

Fitting and transforming data...
Shape of X_train processed tabular data: (40000, 460)
Calculated class weights: {0: 0.5185915054711404, 1: 13.94700139470014}


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ tabular_input (InputLayer)      │ (None, 460)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       118,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 159,233 (622.00 KB)

 Trainable params: 159,233 (622.00 KB)

 Non-trainable params: 0 (0.00 B)


Starting DNN Only model training...
Epoch 1/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.7139 - auc: 0.7832 - loss: 0.5666 - precision: 0.0822 - recall: 0.6862 - val_accuracy: 0.8073 - val_auc: 0.8237 - val_loss: 0.4243 - val_precision: 0.1183 - val_recall: 0.6769
Epoch 2/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7468 - auc: 0.8241 - loss: 0.5119 - precision: 0.0979 - recall: 0.7378 - val_accuracy: 0.8027 - val_auc: 0.8347 - val_loss: 0.4140 - val_precision: 0.1179 - val_recall: 0.6936
Epoch 3/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7761 - auc: 0.8415 - loss: 0.4899 - precision: 0.1099 - recall: 0.7385 - val_accuracy: 0.8500 - val_auc: 0.8372 - val_loss: 0.3222 - val_precision: 0.1437 - val_recall: 0.6407
Epoch 4/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7815 - auc: 0.8551 - loss: 0.4693 - precision: 0.1146 - recall: 0.7573 - val_accuracy: 0.8549 - val_auc: 0.8425 - val_loss: 0.2999 - val_precision: 0.1486 - va